## Text Classification - MLP

## setup

In [ ]:
import torch 
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

corpus = [
    "gậy ông đập lưng ông",
    "có làm mới có ăn"
]
data_size = len(corpus)

# 0: negative - 1: positive
labels = [0, 1]

# Define the max vocabulary size and sequence length
vocab_size = 8
sequence_length = 5

In [3]:
# Initialize the tokenizer and define a trainer
tokenizer = Tokenizer(WordLevel())
tokenizer.pre_tokenizer = Whitespace()
tokenizer.enable_padding(pad_id=1, 
                         pad_token="<pad>", 
                         length=sequence_length)
tokenizer.enable_truncation(max_length=sequence_length)

# Train the tokenizer on your corpus
trainer = WordLevelTrainer(vocab_size=vocab_size, 
                           special_tokens=["<unk>", "<pad>"])
tokenizer.train_from_iterator(corpus, trainer)


# Vectorize the samples
corpus_ids = []
for sentence in corpus:
    output = tokenizer.encode(sentence)
    output = torch.tensor(output.ids, dtype=torch.long)
    corpus_ids.append(output)

In [4]:
for v in corpus_ids:
    print(v)

tensor([4, 3, 0, 6, 3])
tensor([2, 5, 7, 2, 0])


In [ ]:
embedding = nn.Embedding(vocab_size, 4)   # (5, 4)
linear = nn.Linear(4, 3)                  # (5, 3)
flatten = nn.Flatten()
classifier = nn.Linear(5*3, 2)
model = nn.Sequential(embedding, linear, flatten, classifier)

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),
                             lr=0.1)

## Train

In [7]:
inputs = torch.tensor([[4, 3, 0, 6, 3],
                       [2, 5, 7, 2, 0]], dtype=torch.long)
labels = torch.tensor([0, 1], dtype=torch.long)

for _ in range(50):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    print(loss.item())

    loss.backward()
    optimizer.step()

0.5372958779335022
0.45143336057662964
0.3831273019313812
0.32638898491859436
0.27851012349128723
0.23804977536201477
0.20400893688201904
0.17552147805690765
0.1517738401889801
0.13200853765010834
0.1155453473329544
0.10179562866687775
0.09026457369327545
0.0805443823337555
0.07230368256568909
0.0652749240398407
0.05924312025308609
0.0540352389216423
0.04951216280460358
0.04556126892566681
0.04209151864051819
0.03902844339609146
0.036311082541942596
0.033889275044202805
0.03172146901488304
0.029772914946079254
0.02801472693681717
0.02642250992357731
0.024975739419460297
0.023656662553548813
0.022450409829616547
0.02134421095252037
0.020327014848589897
0.019389158114790916
0.018522324040532112
0.01771938055753708
0.01697392389178276
0.01628044806420803
0.01563407853245735
0.015030328184366226
0.014465514570474625
0.013936235569417477
0.01343937311321497
0.012972327880561352
0.012532494030892849
0.012117845937609673
0.01172629464417696
0.011356217786669731
0.011005870066583157
0.01067385

## Check if the model has adready learned sucessfuly

In [8]:
outputs = model(inputs)
print(outputs)
print(torch.softmax(outputs, axis=-1))

tensor([[ 2.3392, -2.2103],
        [-1.8264,  2.7538]], grad_fn=<AddmmBackward0>)
tensor([[0.9895, 0.0105],
        [0.0101, 0.9899]], grad_fn=<SoftmaxBackward0>)
